# METASPACE Python client: examples

Hands-on examples for the [METASPACE Python client](https://metaspace2020.readthedocs.io/en/latest/): find datasets, download annotations and ion images, explore colocalization and off-sample labels, work with ROIs, and submit datasets.
Each section can be run on its own after the shared setup. Sections 7 to 9 need an API key (see the setup).

**Sections**
1. [Setup](#1.-Setup)
2. [Fetch dataset metadata](#2.-Fetch-dataset-metadata)
3. [Fetch dataset annotations](#3.-Fetch-dataset-annotations)
4. [Fetch isotopic images](#4.-Fetch-isotopic-images)
5. [Explore colocalized annotations](#5.-Explore-colocalized-annotations)
6. [Explore off-sample results](#6.-Explore-off-sample-results)
7. [Regions of interest (ROIs)](#7.-Regions-of-interest-(ROIs)) *(API key)*
8. [Submit a dataset](#8.-Submit-a-dataset) *(API key)*
9. [Update dataset databases](#9.-Update-dataset-databases) *(API key)*

---
## 1. Setup

Install the client and connect. All sections share the same `SMInstance`.

**Public data needs no login.** To work with private datasets, or to submit and change datasets (sections 7 to 9), create an API key on your [account page](https://metaspace2020.org/user/me) and pass it as `SMInstance(api_key="...")`. Alternatively, call `sm.save_login()` once: it asks for the key and stores it on your computer. Keep API keys secret, like passwords, and never commit them.

In [ ]:
# Install dependencies (run once per Colab session)
%pip install -q metaspace2020

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from metaspace import SMInstance

sm = SMInstance()  # connects to https://metaspace2020.org

---
## 2. Fetch dataset metadata

Retrieve metadata for all public datasets and explore their properties (ionisation source, MALDI matrix, etc.).

In [ ]:
# List the first three datasets returned by the API
datasets = sm.datasets()
datasets[:3]

In [ ]:
# Fetch a flat metadata table for all datasets
metadata = sm.get_metadata()
metadata.columns.values  # preview available metadata fields

In [ ]:
# Pie chart of ionisation sources across all datasets
ionisation = metadata['MS_Analysis.Ionisation_Source'].value_counts()
ionisation_short = ionisation[:4].copy()
ionisation_short['Other'] = ionisation[4:].sum()
ionisation_short.plot(kind='pie', figsize=(5, 5))
plt.show()

---
## 3. Fetch dataset annotations

Load annotations (metabolites, FDR scores, MSM scores) for a specific dataset and overlay an ion image on its optical image.

In [ ]:
# Select a public mouse-brain dataset by ID
dsid = '2025-04-27_13h26m30s'
ds = sm.dataset(id=dsid)
ds

In [ ]:
# Show which molecular databases were used for annotation
ds.database_details

In [ ]:
# Fetch all results and plot the FDR distribution; filter to 10 % FDR
results = ds.results(database=('HMDB', 'v4'))
results.fdr.value_counts().sort_index().cumsum().plot()
plt.show()

fdr = 0.1
annotations = results[results.fdr <= fdr][['msm']]
annotations.head()

In [ ]:
# Plot all isotope images for the top 5 annotations
limit = 5
plt.figure(figsize=(20, 16))
for ii in range(limit):
    row = results.iloc[ii]
    sf, adduct = row.name
    images = ds.isotope_images(sf, adduct)
    for j, im in enumerate(images):
        ax = plt.subplot(limit, 4, ii * 4 + j + 1)
        ax.axis('off')
        plt.title('[{}{}]+ m/z {}'.format(sf, adduct, images.peak(index=j)))
        plt.imshow(images[j], cmap='viridis')
plt.tight_layout()
plt.show()

In [ ]:
# Overlay an ion image on the dataset's optical image.
# Not every dataset has an optical image, so we use one that does.
ds_oi = sm.dataset(id='2022-08-05_17h28m56s')
oi = ds_oi.optical_images()

formula, adduct = ds_oi.results(database=('BraChemDB', '2018-01'), fdr=0.1).index[0]
image = ds_oi.isotope_images(formula, adduct)[0]

plt.figure()
plt.imshow(oi[0])
plt.imshow(
    np.asarray(oi.ion_image_to_optical(image)).sum(axis=2),
    cmap='viridis', alpha=0.5
)
plt.show()

---
## 4. Fetch isotopic images

Download isotope images individually or in bulk, then compute a pixel-level Pearson correlation matrix across all annotations.

In [ ]:
# Reuse the same dataset; get annotations at 10 % FDR
ds = sm.dataset(id='2025-04-27_13h26m30s')
results = ds.results()
annotations = results[results.fdr <= 0.1][['msm']]
annotations.head()

In [ ]:
# Bulk-download one image per annotation and compute the Pearson correlation matrix
isotope_image_sets = ds.all_annotation_images(fdr=0.10, only_first_isotope=True, scale_intensity=False)

images_matrix = np.array([imgs[0].flatten() for imgs in isotope_image_sets])
labels = [(imgs.formula, imgs.adduct) for imgs in isotope_image_sets]

corr = np.corrcoef(images_matrix)
corr_df = pd.DataFrame(corr, index=labels, columns=labels)
corr_df

---
## 5. Explore colocalized annotations

Query annotations that are spatially colocalized with a target molecule and visualise the top matches.

In [ ]:
ds = sm.dataset(id='2025-04-27_13h26m30s')
ds

In [ ]:
# Helper: download the first isotope image for each (formula, adduct) pair
def fetch_images(ds, annotations):
    ann_images = {}
    for formula, adduct in annotations:
        image = ds.isotope_images(formula, adduct, only_first_isotope=True)[0]
        ann_images[(formula, adduct)] = image
    return ann_images

In [ ]:
# Fetch results colocalized with a specific lipid at 10 % FDR
database = 'HMDB-v4'
fdr = 0.1

formula, adduct = 'C44H86NO8P', '+K'
coloc_with = f'{formula}{adduct}+'

df = ds.results(database=database, fdr=fdr, coloc_with=coloc_with)
df

In [ ]:
# Download and plot the top 10 colocalized images
top_n = 10
top_coloc_anns = df.head(top_n).index.tolist()
most_coloc_images = fetch_images(ds, top_coloc_anns)

fig, axes = plt.subplots(5, 2, figsize=(15, 30))
axes = axes.flatten()
for i, row in enumerate(df.head(top_n).itertuples()):
    formula, adduct = row.Index
    image = most_coloc_images[(formula, adduct)]
    axes[i].imshow(image)
    axes[i].set_title(f'{formula}, {adduct}: {row.colocCoeff:.4f}')
plt.tight_layout()
plt.show()

---
## 6. Explore off-sample results

Retrieve pre-computed off-sample predictions from METASPACE and visualise which annotations are classified as on-sample vs. off-sample.

In [ ]:
ds = sm.dataset(id='2025-04-27_13h26m30s')
fdr = 0.1
moldb_name = 'HMDB-v4'


In [ ]:
# Download all annotation images
ds_images = ds.all_annotation_images(fdr=fdr, database=moldb_name)

ann_docs = []
for img in ds_images:
    ann_docs.append({
        'formula': img._sf,
        'adduct': img._adduct,
        'image': next(iter(img)),
        'image_url': img._urls[0],
    })

In [ ]:
# Fetch off-sample predictions stored in the database
anns = ds.annotations(
    fdr=fdr, database=moldb_name,
    return_vals=('sumFormula', 'adduct', 'offSample', 'offSampleProb')
)
off_sample_preds = {(formula, adduct): off_sample for (formula, adduct, *off_sample) in anns}

for ann in ann_docs:
    label, prob = off_sample_preds[(ann['formula'], ann['adduct'])]
    ann['label'] = label
    ann['prob'] = prob

In [ ]:
# Visualise a grid of annotation images with their off-sample labels
def plot_images(images, labels, limit=20, coln=4):
    if not images:
        print('No annotations in this group.')
        return
    limit = min(limit, len(images))
    rown = (limit - 1) // coln + 1
    fig, axes = plt.subplots(rown, coln, figsize=(5 * coln, 5 * rown))
    for i in range(rown):
        for j in range(coln):
            idx = i * coln + j
            if idx < limit:
                ax = axes[i, j] if axes.ndim > 1 else axes[j]
                ax.imshow(images[idx], cmap='viridis')
                ax.set_title(labels[idx])
    plt.show()


def ann_to_images_labels(anns):
    images = [d['image'] for d in anns]
    labels = [
        '{} {}\nlabel: {}, off prob: {:.3f}'.format(
            d['formula'], d['adduct'],
            'off' if d['label'] else 'on', d['prob']
        )
        for d in anns
    ]
    return images, labels

In [ ]:
# Off-sample annotations (off-sample probability > 0.5)
off_ann_docs = [d for d in ann_docs if d['prob'] > 0.5]
images, labels = ann_to_images_labels(off_ann_docs)
plot_images(images, labels, limit=8)

In [ ]:
# On-sample annotations (off-sample probability < 0.5)
on_ann_docs = [d for d in ann_docs if d['prob'] < 0.5]
images, labels = ann_to_images_labels(on_ann_docs)
plot_images(images, labels, limit=8)

In [ ]:
# Borderline annotations (off-sample probability between 0.4 and 0.6)
border_ann_docs = [d for d in ann_docs if 0.4 < d['prob'] < 0.6]
images, labels = ann_to_images_labels(border_ann_docs)
plot_images(images, labels, limit=8)

---
## 7. Regions of interest (ROIs)

ROIs are regions drawn on an ion image in the METASPACE web app, for example tumor and normal tissue. ROIs belong to a user, so these calls need an API key and a dataset on which you have drawn or imported ROIs.

> **Needs an API key.** The cells in this section log in to your METASPACE account and change your own data. They are not meant for "Run all": read them first and run them deliberately.

In [ ]:
sm = SMInstance(api_key="")  # paste your API key (keep it out of shared notebooks)
ds_id = "<your-dataset-id>"

# ROIs saved on the dataset, each with an id, name and GeoJSON string
rois = sm.get_rois(ds_id)
[(roi["id"], roi["name"]) for roi in rois]

In [ ]:
# Export all ROIs as a GeoJSON FeatureCollection, the same format as the web app's ROI export.
# Vertices are ion image pixel coordinates (x = column, y = row).
geojson = sm.export_rois(ds_id, path="rois.geojson")
geojson["metadata"]

In [ ]:
# Check a GeoJSON file against a dataset before importing it, then import it.
# Importing always adds new ROIs; it never replaces or removes existing ones.
report = sm.validate_roi_geojson(ds_id, "rois.geojson")
print(report["valid"], report["errors"])

if report["valid"]:
    sm.import_rois(ds_id, "rois.geojson")

The exported GeoJSON can be used to label pixels of a dataset downloaded with the [METASPACE converter](https://github.com/metaspace2020/metaspace-converter).

---
## 8. Submit a dataset

Upload an imzML/ibd file pair to METASPACE with the required metadata and a list of annotation databases.

> **Needs an API key.** The cells in this section log in to your METASPACE account and change your own data. They are not meant for "Run all": read them first and run them deliberately.

In [ ]:
# Paths to your own imzML and ibd files
imzml_fn = "path/to/your_dataset.imzML"
ibd_fn = "path/to/your_dataset.ibd"

In [ ]:
# Login to API. Need to add your API key
sm = SMInstance(api_key="")

In [ ]:
# Dataset name and required metadata
dataset_name = 'My_example_dataset'
metadata = {
    'Data_Type': 'Imaging MS',
    'Sample_Information': {
        'Organism': 'Test organism',
        'Organism_Part': 'Test part',
        'Condition': 'Test condition',
        'Sample_Growth_Conditions': 'N/A',
    },
    'Sample_Preparation': {
        'Sample_Stabilisation': 'None',
        'Tissue_Modification': 'None',
        'MALDI_Matrix': '2,5-dihydroxyacetophenone (DHA)',
        'MALDI_Matrix_Application': 'None',
        'Solvent': 'none',
    },
    'MS_Analysis': {
        'Polarity': 'Positive',
        'Ionisation_Source': 'MALDI',
        'Analyzer': 'Orbitrap',
        'Detector_Resolving_Power': {'mz': 200, 'Resolving_Power': 140000},
        'Pixel_Size': {'Xaxis': 10, 'Yaxis': 10},
    },
}

In [ ]:
# Print available databases to help select the ones to annotate against
print(sm.databases())

In [ ]:
# Databases can be specified by integer ID or as (name, version) tuples
databases = [
    22,                     # HMDB v4
    ('CoreMetabolome', 'v3'),   # (name, version) form is also accepted
]

is_public = True  # set False to keep the dataset private

sm.submit_dataset(imzml_fn, ibd_fn, dataset_name, metadata, is_public, databases)

---
## 9. Update dataset databases

Add or change the molecular databases used to annotate an existing dataset, either for a single dataset or for every dataset in a project.

> **Needs an API key.** The cells in this section log in to your METASPACE account and change your own data. They are not meant for "Run all": read them first and run them deliberately.

In [ ]:
from pprint import pprint

def databases_as_tuples(database_details):
    return [(db['name'], db['version']) for db in database_details]

# Inspect the current databases for a dataset
dataset_id = '<your-dataset-id>'
dataset = sm.dataset(id=dataset_id)
pprint(databases_as_tuples(dataset.database_details))

In [ ]:
# Replace the annotation databases for the dataset
new_databases = [('ChEBI', '2018-01'), ('LipidMaps', '2017-12-12')]

ds = sm.dataset(id=dataset_id)
sm.update_dataset_dbs(dataset_id, new_databases, ds.adducts)